In [ ]:
from unsloth import FastVisionModel
import torch

In [ ]:
base_model = "unsloth/Qwen3.5-0.8B"

model, tokenizer = FastVisionModel.from_pretrained(
    base_model,
    load_in_4bit=False,
    use_gradient_checkpointing="unsloth"
)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_rslora=False,
    loftq_config=None
)

In [ ]:
import sqlite3

db_path = "/home/rngo/code/avg-dev/projects/ai/pokemon-ai/dataset.db"

db_connection = sqlite3.connect(db_path)
db_connection.row_factory = sqlite3.Row

In [ ]:
cursor = db_connection.cursor()

results = [
    dict(r) for r in cursor.execute("SELECT * FROM annotations WHERE task_type = 'bbox'").fetchall()
]

In [ ]:
results[:3]

In [ ]:
import json
from PIL import Image


base_image_path = "/home/rngo/code/avg-dev/projects/ai/pokemon-ai/data"

def convert_to_conversation(example):
    label = {
        "bounding_boxes": json.loads(example["bounding_boxes"])
    }

    conversation = [
        {
            "role": "system",
            "content": [
                { "type": "text", "text": example["system_prompt"] }
            ]
        },
        {
            "role": "user",
            "content": [
                { "type": "image", "image": Image.open(f"{base_image_path}/{example["image_filename"]}") },
                { "type": "text", "text": example["instruction"] }
            ]
        },
        {
            "role": "assistant",
            "content": [
                { "type": "text", "text": json.dumps(label)}
            ]
        }
    ]

    return {"messages": conversation}

In [ ]:
train_dataset = [
    convert_to_conversation(example) for example in \
        list(filter(lambda x: x["validation_set"] == 0, results))
]

eval_dataset = [
    convert_to_conversation(example) for example in \
        list(filter(lambda x: x["validation_set"] == 1, results))
]

In [ ]:
len(results), len(train_dataset), len(eval_dataset)

In [ ]:
FastVisionModel.for_inference(model)

conv = eval_dataset[0]["messages"][:2]

input_text = tokenizer.apply_chat_template(
    conv,
    add_generation_prompt = True
)

test_image = conv[1]["content"][0]["image"]

inputs = tokenizer(
    test_image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt"
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.1,
    min_p=0.1
)


In [ ]:
from PIL import ImageDraw
from IPython.display import clear_output

def draw_box(image, coords):
    clear_output()

    copied = image.copy()

    draw = ImageDraw.Draw(copied)

    x, y, width, height = coords

    rect = (x, y, x + width, y + height)

    draw.rectangle(rect, outline="green", width=2)

    display(copied)

In [ ]:
draw_box(test_image, (195, 178, 39, 50))

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        eval_strategy="steps",
        eval_steps=10,
        warmup_ratio=0.03,
        num_train_epochs=10,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        output_dir="outputs",
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048
    )
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
FastVisionModel.for_inference(model)

conv = eval_dataset[0]["messages"][:2]

input_text = tokenizer.apply_chat_template(
    conv,
    add_generation_prompt = True
)

test_image = conv[1]["content"][0]["image"]

inputs = tokenizer(
    test_image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt"
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.1,
    min_p=0.1
)


In [ ]:
draw_box(test_image, (195, 178, 39, 50))

In [ ]:
# save the fine tuned model
model.save_pretrained_merged("outputs/pokemon_player_recognizer", tokenizer)